In [1]:
!wget https://figshare.com/ndownloader/files/62948443

--2026-03-19 16:50:42--  https://figshare.com/ndownloader/files/62948443
Resolving figshare.com (figshare.com)... 34.252.200.63, 52.49.4.28, 34.255.171.171, ...
Connecting to figshare.com (figshare.com)|34.252.200.63|:443... connected.
HTTP request sent, awaiting response... 403 Forbidden
2026-03-19 16:50:43 ERROR 403: Forbidden.



In [ ]:
import pickle
from tqdm import tqdm
with open("dataset_for_5cross.dataset", "rb") as f:
    data = pickle.load(f)
data.head()

In [ ]:
positions_mut0_list = []
positions_mut1_list = []
for i in tqdm(range(len(data))):
    sample = data.iloc[i]
    positions_mut0 = []
    positions_mut1 = []
    for j in range(len(sample['positions_mut0'])):
        pos = sample['positions_mut0'][j].split('-')
        positions_mut0.append((int(pos[0]), int(pos[1])))
        pos = sample['positions_mut1'][j].split('-')
        positions_mut1.append((int(pos[0]), int(pos[1])))
    positions_mut0_list.append(positions_mut0)
    positions_mut1_list.append(positions_mut1)
data['positions_mut0'] = positions_mut0_list
data['positions_mut1'] = positions_mut1_list
data.head()

In [ ]:
len_range = 10
from tqdm import tqdm
import pandas as pd

valid_indices = []

for i in tqdm(range(len(data))):
    sample = data.iloc[i]

    center0 = int((sample['positions_mut0'][0][0] + sample['positions_mut0'][-1][1]) / 2)
    if center0 - len_range > sample['positions_mut0'][0][0]:
        continue
    if center0 + len_range + 1 < sample['positions_mut0'][-1][1]:
        continue

    center1 = int((sample['positions_mut1'][0][0] + sample['positions_mut1'][-1][1]) / 2)
    if center1 - len_range > sample['positions_mut1'][0][0]:
        continue
    if center1 + len_range + 1 < sample['positions_mut1'][-1][1]:
        continue

    valid_indices.append(i)

data = data.iloc[valid_indices].reset_index(drop=True)
print(f"{len(data)}")

In [ ]:
data = data[
    data['mut0'].str.len() < 3000
]
data = data[
    data['par0'].str.len() < 1000
]
data = data[data['label'] != 4]
data.shape

In [ ]:
#!source /usr/local/Ascend/ascend-toolkit/set_env.sh
import torch

import torch.nn as nn
import math
from functools import partial
from esm.modules import ContactPredictionHead, ESM1bLayerNorm, RobertaLMHead, TransformerLayer, MultiheadAttention 
import esm
from typing import Union

class ESMModelWrapper(nn.Module):
    def __init__(self, model):
        super(ESMModelWrapper, self).__init__()
        self.model = model

    def forward(self, batch_tokens, repr_layers=[33], return_contacts=False):
        return self.model(batch_tokens, repr_layers=repr_layers, return_contacts=return_contacts)


class ESMFeatureEncoder(nn.Module):
    def __init__(self):
        super(ESMFeatureEncoder, self).__init__()
        self.device = 'cuda:0'
        self.model, self.alphabet = esm.pretrained.esm2_t33_650M_UR50D()
        self.model.to(self.device)
        # self.model = torch.nn.DataParallel(self.model, device_ids=[0, 1, 2])
        self.model.eval()  # Set the model to evaluation mode
        self.batch_converter = self.alphabet.get_batch_converter()
        self.padding_idx = self.alphabet.padding_idx
        # Wrap the model with the ESMModelWrapper
        self.model = ESMModelWrapper(self.model)

    def encode(self, sequences):
        batch_labels, batch_strs, batch_tokens = self.batch_converter([(str(0), sequence) for sequence in sequences])
        batch_tokens = batch_tokens.to(self.device)
        batch_mask = batch_tokens.eq(self.padding_idx)
        # print(batch_tokens.shape)
        with torch.no_grad():
            results = self.model(batch_tokens, repr_layers=[33], return_contacts=False)
        token_representations = results['representations'][33]
        # print(results['representations'][33].mean(dim=1).unsqueeze(1).shape)
        return token_representations,batch_mask



In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

from tqdm import tqdm
import numpy as np
import os

#ESM_feature ---need to run for the first time
esm_model = ESMFeatureEncoder()
for i in tqdm(range(len(data))):
    sample = data.iloc[i]
    file_path = f'../data/middlefile/ESM_feature/{sample["#Feature AC"]}/'
    if os.path.exists(file_path):
        continue
    embedding, mask = esm_model.encode([sample['mut0']])
    mut0 = embedding.detach().cpu().numpy()
    embedding, mask = esm_model.encode([sample['mut1']])
    mut1 = embedding.detach().cpu().numpy()
    embedding, mask = esm_model.encode([sample['par0']])
    par0 = embedding.detach().cpu().numpy()
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    np.save(file_path + 'mut0.npy', mut0)
    np.save(file_path + 'mut1.npy', mut1)
    np.save(file_path + 'par0.npy', par0)

In [ ]:
import numpy as np
def positional_encoding(max_len, d_model):

    position = torch.arange(max_len, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(np.log(10000.0) / d_model))
    
    pe = torch.zeros(max_len, d_model)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    pe.requires_grad = False
    return pe

len_range = 10
position_embedding = positional_encoding(2 * (2 * len_range + 1) + 2 , 1280).numpy()
position_embedding[2 * len_range + 3:, :].shape, position_embedding.shape

In [ ]:
in_range = []
out_range = []
for i in tqdm(range(len(data))):
    usable = True
    sample = data.iloc[i]
    for j in range(len(sample['positions_mut0'])):
        center = int((sample['positions_mut0'][j][0] + sample['positions_mut0'][j][1]) / 2)
        if sample['positions_mut0'][j][0] < center - len_range or sample['positions_mut0'][j][1] > center + len_range:
            usable = False

        center = int((sample['positions_mut1'][j][0] + sample['positions_mut1'][j][1]) / 2)
        if sample['positions_mut1'][j][0] < center - len_range or sample['positions_mut1'][j][1] > center + len_range:
            usable = False

    if usable:
        in_range.append(sample)
    else:
        out_range.append(sample)

data = pd.DataFrame(in_range)
len(in_range), len(out_range)

In [ ]:
mut0_f_list, mut1_f_list, par0_f_list = [], [], []
for i in tqdm(range(len(data))):
    sample = data.iloc[i]
    correct = True
    file_path = f'../data/middlefile/ESM_feature/{sample["#Feature AC"]}/'
    mut0_feature = np.load(file_path + 'mut0.npy', allow_pickle=True).squeeze(0)

    mut1_feature = np.load(file_path + 'mut1.npy', allow_pickle=True).squeeze(0)

    par0_feature = np.load(file_path + 'par0.npy', allow_pickle=True).squeeze(0)

    parts_mut0 = []
    parts_mut1 = []
    parts_mut0.append(np.expand_dims(mut0_feature.mean(axis=0), axis = 0) + position_embedding[:1, :])
    parts_mut1.append(np.expand_dims(mut1_feature.mean(axis=0), axis = 0) + position_embedding[2 * len_range + 2 : 2 * len_range + 3, :])
    # for j in range(len(sample['positions_mut0'])):
    #     # if sample['mut1'][sample['positions_mut1'][j][0] : sample['positions_mut1'][j][1]] != sample['Resulting sequence'][j]
        
    #     center = int((sample['positions_mut0'][j][0] + sample['positions_mut0'][j][1]) / 2)

    #     # start = max(center - len_range, 0)
    #     start = max(center - len_range + 1, 0)
    #     part_mut0 = mut0_feature[start: center + len_range + 1 + 1, :]
    #     if start == 0:
    #         positions_ed = position_embedding[1 : 2 * len_range + 1 + 1, : ][-part_mut0.shape[0] : , :]
    #     else:
    #         positions_ed = position_embedding[1 : 2 * len_range + 1 + 1, : ][ : part_mut0.shape[0] , :]
    #     parts_mut0.append(part_mut0 + positions_ed)


    #     center = int((sample['positions_mut1'][j][0] + sample['positions_mut1'][j][1]) / 2)

    #     # start = max(center - len_range, 0)
    #     start = max(center - len_range + 1, 0)
    #     part_mut1 = mut1_feature[start: center + len_range + 1 + 1, :]
    #     if start == 0:
    #         positions_ed = position_embedding[3 + 2 * len_range : , : ][-part_mut1.shape[0] : , :]
    #     else:
    #         positions_ed = position_embedding[3 + 2 * len_range : , : ][ : part_mut1.shape[0] , :]
    #     parts_mut1.append(part_mut1 + positions_ed)
    center = int((sample['positions_mut0'][0][0] + sample['positions_mut0'][-1][1]) / 2)
    # if center - len_range > sample['positions_mut0'][0][0] :
    #     continue
    # if center + len_range + 1 < sample['positions_mut0'][-1][1] :
    #     continue


    start = max(center - len_range + 1, 0)
    part_mut0 = mut0_feature[start: center + len_range + 1 + 1, :]
    if start == 0:
        positions_ed = position_embedding[1 : 2 * len_range + 1 + 1, : ][-part_mut0.shape[0] : , :]
    else:
        positions_ed = position_embedding[1 : 2 * len_range + 1 + 1, : ][ : part_mut0.shape[0] , :]
    part_mut0 = part_mut0 + positions_ed
    parts_mut0.append(part_mut0)
    


    center = int((sample['positions_mut1'][0][0] + sample['positions_mut1'][-1][1]) / 2)
    # if center - len_range > sample['positions_mut1'][0][0] :
    #     continue
    # if center + len_range + 1 < sample['positions_mut1'][-1][1] :
    #     continue

    
    start = max(center - len_range + 1, 0)
    part_mut1 = mut1_feature[start: center + len_range + 1 + 1, :]
    if start == 0:
        positions_ed = position_embedding[3 + 2 * len_range : , : ][-part_mut1.shape[0] : , :]
    else:
        positions_ed = position_embedding[3 + 2 * len_range : , : ][ : part_mut1.shape[0] , :]
    part_mut1 = part_mut1 + positions_ed
    parts_mut1.append(part_mut1)
    
    # result = np.concatenate(all_arrays, axis=0)
    mut0_f_list.append(np.concatenate(parts_mut0, axis=0))
    mut1_f_list.append(np.concatenate(parts_mut1, axis=0))
    par0_f_list.append(par0_feature)
        # if sample['mut0'][start: center + len_range + 1][len_range] != sample['Original sequence'][j]:
        #     correct = False
        #     break
    
data['mut0_f'] = mut0_f_list
data['mut1_f'] = mut1_f_list
data['par0_f'] = par0_f_list

In [ ]:
import pandas as pd

# df = pd.read_pickle(data_path)
train_df = data
new_df1 = train_df[train_df['label'] == 1].copy()
new_df1['mut0'], new_df1['mut1'] = new_df1['mut1'], new_df1['mut0']
# new_df1['mut0_f'], new_df1['mut1_f'] = new_df1['mut1_f'], new_df1['mut0_f']
mut0_f_list, mut1_f_list = [], []
for i in tqdm(range(len(new_df1))):
    sample = new_df1.iloc[i]
    correct = True
    file_path = f'../../data/middlefile/ESM_feature/{sample["#Feature AC"]}/'
    mut0_feature = np.load(file_path + 'mut1.npy', allow_pickle=True).squeeze(0)

    mut1_feature = np.load(file_path + 'mut0.npy', allow_pickle=True).squeeze(0)


    parts_mut0 = []
    parts_mut1 = []
    parts_mut0.append(np.expand_dims(mut0_feature.mean(axis=0), axis = 0) + position_embedding[:1, :])
    parts_mut1.append(np.expand_dims(mut1_feature.mean(axis=0), axis = 0) + position_embedding[2 * len_range + 2 : 2 * len_range + 3, :])

    center = int((sample['positions_mut1'][0][0] + sample['positions_mut1'][-1][1]) / 2)

    start = max(center - len_range + 1, 0)
    part_mut0 = mut0_feature[start: center + len_range + 1 + 1, :]
    if start == 0:
        positions_ed = position_embedding[1 : 2 * len_range + 1 + 1, : ][-part_mut0.shape[0] : , :]
    else:
        positions_ed = position_embedding[1 : 2 * len_range + 1 + 1, : ][ : part_mut0.shape[0] , :]
    part_mut0 = part_mut0 + positions_ed
    parts_mut0.append(part_mut0)

    
    center = int((sample['positions_mut0'][0][0] + sample['positions_mut0'][-1][1]) / 2)

    start = max(center - len_range + 1, 0)
    part_mut1 = mut1_feature[start: center + len_range + 1 + 1, :]
    if start == 0:
        positions_ed = position_embedding[3 + 2 * len_range : , : ][-part_mut1.shape[0] : , :]
    else:
        positions_ed = position_embedding[3 + 2 * len_range : , : ][ : part_mut1.shape[0] , :]
    part_mut1 = part_mut1 + positions_ed
    parts_mut1.append(part_mut1)
    
    # result = np.concatenate(all_arrays, axis=0)
    mut0_f_list.append(np.concatenate(parts_mut0, axis=0))
    mut1_f_list.append(np.concatenate(parts_mut1, axis=0))
    
new_df1['mut0_f'] = mut0_f_list
new_df1['mut1_f'] = mut1_f_list
new_df1['label'] = 3


new_df2 = train_df[train_df['label'] == 3].copy()
new_df2['mut0'], new_df2['mut1'] = new_df2['mut1'], new_df2['mut0']
# new_df2['mut0_f'], new_df2['mut1_f'] = new_df2['mut1_f'], new_df2['mut0_f']
mut0_f_list, mut1_f_list = [], []
for i in tqdm(range(len(new_df2))):
    sample = new_df2.iloc[i]
    correct = True
    file_path = f'../../data/middlefile/ESM_feature/{sample["#Feature AC"]}/'
    mut0_feature = np.load(file_path + 'mut1.npy', allow_pickle=True).squeeze(0)

    mut1_feature = np.load(file_path + 'mut0.npy', allow_pickle=True).squeeze(0)


    parts_mut0 = []
    parts_mut1 = []
    parts_mut0.append(np.expand_dims(mut0_feature.mean(axis=0), axis = 0) + position_embedding[:1, :])
    parts_mut1.append(np.expand_dims(mut1_feature.mean(axis=0), axis = 0) + position_embedding[2 * len_range + 2 : 2 * len_range + 3, :])

    center = int((sample['positions_mut1'][0][0] + sample['positions_mut1'][-1][1]) / 2)

    start = max(center - len_range + 1, 0)
    part_mut0 = mut0_feature[start: center + len_range + 1 + 1, :]
    if start == 0:
        positions_ed = position_embedding[1 : 2 * len_range + 1 + 1, : ][-part_mut0.shape[0] : , :]
    else:
        positions_ed = position_embedding[1 : 2 * len_range + 1 + 1, : ][ : part_mut0.shape[0] , :]
    part_mut0 = part_mut0 + positions_ed
    parts_mut0.append(part_mut0)

    
    center = int((sample['positions_mut0'][0][0] + sample['positions_mut0'][-1][1]) / 2)

    start = max(center - len_range + 1, 0)
    part_mut1 = mut1_feature[start: center + len_range + 1 + 1, :]
    if start == 0:
        positions_ed = position_embedding[3 + 2 * len_range : , : ][-part_mut1.shape[0] : , :]
    else:
        positions_ed = position_embedding[3 + 2 * len_range : , : ][ : part_mut1.shape[0] , :]
    part_mut1 = part_mut1 + positions_ed
    parts_mut1.append(part_mut1)
    
    # result = np.concatenate(all_arrays, axis=0)
    mut0_f_list.append(np.concatenate(parts_mut0, axis=0))
    mut1_f_list.append(np.concatenate(parts_mut1, axis=0))
    
new_df2['mut0_f'] = mut0_f_list
new_df2['mut1_f'] = mut1_f_list
new_df2['label'] = 1

new_df3 = train_df[train_df['label'] == 2].copy()
new_df3['mut0'], new_df3['mut1'] = new_df3['mut1'], new_df3['mut0']
# new_df3['mut0_f'], new_df3['mut1_f'] = new_df3['mut1_f'], new_df3['mut0_f']

mut0_f_list, mut1_f_list = [], []
for i in tqdm(range(len(new_df3))):
    sample = new_df3.iloc[i]
    correct = True
    file_path = f'../../data/middlefile/ESM_feature/{sample["#Feature AC"]}/'
    mut0_feature = np.load(file_path + 'mut1.npy', allow_pickle=True).squeeze(0)

    mut1_feature = np.load(file_path + 'mut0.npy', allow_pickle=True).squeeze(0)

    parts_mut0 = []
    parts_mut1 = []
    parts_mut0.append(np.expand_dims(mut0_feature.mean(axis=0), axis = 0) + position_embedding[:1, :])
    parts_mut1.append(np.expand_dims(mut1_feature.mean(axis=0), axis = 0) + position_embedding[2 * len_range + 2 : 2 * len_range + 3, :])

    center = int((sample['positions_mut1'][0][0] + sample['positions_mut1'][-1][1]) / 2)

    start = max(center - len_range + 1, 0)
    part_mut0 = mut0_feature[start: center + len_range + 1 + 1, :]
    if start == 0:
        positions_ed = position_embedding[1 : 2 * len_range + 1 + 1, : ][-part_mut0.shape[0] : , :]
    else:
        positions_ed = position_embedding[1 : 2 * len_range + 1 + 1, : ][ : part_mut0.shape[0] , :]
    part_mut0 = part_mut0 + positions_ed
    parts_mut0.append(part_mut0)

    
    center = int((sample['positions_mut0'][0][0] + sample['positions_mut0'][-1][1]) / 2)

    start = max(center - len_range + 1, 0)
    part_mut1 = mut1_feature[start: center + len_range + 1 + 1, :]
    if start == 0:
        positions_ed = position_embedding[3 + 2 * len_range : , : ][-part_mut1.shape[0] : , :]
    else:
        positions_ed = position_embedding[3 + 2 * len_range : , : ][ : part_mut1.shape[0] , :]
    part_mut1 = part_mut1 + positions_ed
    parts_mut1.append(part_mut1)
    
    # result = np.concatenate(all_arrays, axis=0)
    mut0_f_list.append(np.concatenate(parts_mut0, axis=0))
    mut1_f_list.append(np.concatenate(parts_mut1, axis=0))
    
new_df3['mut0_f'] = mut0_f_list
new_df3['mut1_f'] = mut1_f_list
new_df3['label'] = 2
df = pd.concat([train_df, new_df1, new_df2, new_df3], ignore_index=True)

print(data.shape)
data.head()

In [ ]:
import torch

import torch.nn as nn
import math
from functools import partial
from esm.modules import ContactPredictionHead, ESM1bLayerNorm, RobertaLMHead, TransformerLayer, MultiheadAttention 
import esm
from typing import Union

class GroundingAttention(nn.Module):
    def __init__(self, dim, num_heads=4, qkv_bias=True,
                 attn_drop=0., proj_drop=0.):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim**-0.5

        self.kv = nn.Linear(dim, dim*2, bias=qkv_bias)
        self.q = nn.Linear(dim, dim, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        # self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x, r):
        B, N, C = x.shape
        B_, N_, C_ = r.shape

        kv = self.kv(r).reshape(B_, N_, 2, self.num_heads, C_ //
                                self.num_heads).permute(2, 0, 3, 1, 4)
        k, v = kv.unbind(0)
        q = self.q(x).reshape(B, N, self.num_heads, C //
                              self.num_heads).permute(0, 2, 1, 3)

        attn = (q @ k.transpose(-2, -1)) * self.scale  # (B, heads, N, N_)
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        # x = self.proj(x)
        x = self.proj_drop(x)
        return x

class FFN(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(FFN, self).__init__()
        self.relu = nn.ReLU()
        self.linear1 = nn.Linear(input_dim, hidden_dim)
        self.linear2 = nn.Linear(hidden_dim, input_dim)

    def forward(self, x):

        residual = x

        x = self.linear1(x)
        x = self.relu(x)
        x = self.linear2(x)

        x = residual + x
        return x

class bertlayer(nn.Module):
    def __init__(self, embeddingdim, hidden_dim, num_head = 16):
        super(bertlayer, self).__init__()
        self.atte_norm = ESM1bLayerNorm(embeddingdim)
        self.ffn_norm = ESM1bLayerNorm(embeddingdim)
        self.atte = torch.nn.MultiheadAttention(embed_dim = embeddingdim, num_heads = num_head, dropout = 0.0)
        self.ffn = FFN(embeddingdim, hidden_dim)
    def forward(self, x, x_padding_mask):
        residual = x
        x = self.atte_norm(x)
        x, _ = self.atte(x, x, x, key_padding_mask = x_padding_mask)
        x = x + residual
        x = x + self.ffn(self.ffn_norm(x))
        return x


class InteractionBlock(nn.Module):
    def __init__(self, embed_dim, ffn_dim, BertLayerNorm = ESM1bLayerNorm, attention_heads = 16, add_bias_kv = True, use_rotary_embeddings = True):
        super(InteractionBlock, self).__init__()
        # self.injector_query_norm = norm_layer(embedding_dim)
        # self.injector_kv_norm = norm_layer(embedding_dim)
        # self.extractor_query_norm = norm_layer(embedding_dim)
        # self.extractor_kv_norm = norm_layer(embedding_dim)
        # self.extractor_norm = norm_layer(embedding_dim)
        # self.injector = GroundingAttention(embedding_dim)
        # self.block = GroundingAttention(embedding_dim)
        # self.extractor = GroundingAttention(embedding_dim)
        # self.extractor_ffn = FFN(embedding_dim * ffn_dim, ffn_dim_rate * embedding_dim * ffn_dim)
        self.attention_heads = attention_heads
        self.embed_dim = embed_dim
        self.ffn_dim = ffn_dim
        self.injector_q_norm = BertLayerNorm(embed_dim)
        self.injector_kv_norm = BertLayerNorm(embed_dim)
        # self.injector = GroundingAttention(embed_dim)

        self.injector = torch.nn.MultiheadAttention(embed_dim = embed_dim, num_heads = attention_heads, dropout = 0.0)
        self.block = bertlayer(embed_dim, embed_dim * 4)
        self.extractor_q_norm = BertLayerNorm(embed_dim)
        self.extractor_kv_norm = BertLayerNorm(embed_dim)
        self.extractor = torch.nn.MultiheadAttention(embed_dim = embed_dim, num_heads = attention_heads, dropout = 0.0)
        self.ffn = FFN(embed_dim, ffn_dim)
    def forward(self, x, r, x_attn_padding_mask=None, r_attn_padding_mask = None, need_head_weights=False):
        # x = self.injector(self.injector_query_norm(x), self.injector_kv_norm(r)) + x
        # x = self.block(x, x)
        # r = self.extractor(self.extractor_query_norm(r), self.extractor_kv_norm(x)) + r
        # r = r + self.extractor_ffn(self.extractor_norm(r))
        
        
        # x, _ = self.injector_attention(
        #     query=self.injector_q_norm(x),
        #     key=self.injector_kv_norm(r),
        #     value=self.injector_kv_norm(r),
        #     key_padding_mask=self_attn_padding_mask,
        #     need_weights=True,
        #     need_head_weights=need_head_weights,
        #     attn_mask=self_attn_mask,
        # )
        # print(self.injector_q_norm(x).shape)
        # print(self.injector_q_norm(r).shape)
        
        
        # x = x + self.injector(self.injector_q_norm(x),self.injector_kv_norm(r))
        residual_x = x
        residual_r = r
        # x = self.injector_q_norm(x)
        # r = self.injector_kv_norm(r)
        x = x.transpose(0, 1)
        r = r.transpose(0, 1)
        x = self.injector_q_norm(x)
        r = self.injector_kv_norm(r)
        # print("r.shape")
        # print(r.shape)
        # print("r_attn_padding_mask.shape")
        # print(r_attn_padding_mask.shape)
        # print("x.shape")
        # print(x.shape)
        x, attn = self.injector(x, r, r, key_padding_mask=r_attn_padding_mask )
        x = x.transpose(0, 1)
        x = x + residual_x
        r = residual_r
        x = x.transpose(0, 1)
        x = self.block(x, x_attn_padding_mask)
        
        # r = r + self.extractor(self.extractor_q_norm(r),self.extractor_kv_norm(x))
        residual_r = r
        residual_x = x.transpose(0, 1)
        # x = self.extractor_kv_norm(x)
        # r = self.extractor_q_norm(r)
        # x = x.transpose(0, 1)
        r = r.transpose(0, 1)
        # print(r.shape)
        # print(x.shape)
        # print(x_attn_padding_mask.shape)
        x = self.extractor_kv_norm(x)
        r = self.extractor_q_norm(r)
        r, attn = self.extractor(r, x, x, key_padding_mask=x_attn_padding_mask)
        r = r.transpose(0, 1)
        r = r + residual_r
        
        r = self.ffn(r)
        return residual_x, r

import esm
class DynamicFeatureSelector(nn.Module):
    def __init__(self, input_size, num_features, num_layers=4):
        super(DynamicFeatureSelector, self).__init__()
        

        self.hidden_layers = []
        self.batch_norm_layers = []  
        current_size = input_size
        decrement = (input_size - num_features) // num_layers  

        for _ in range(num_layers):
            if current_size <= num_features:
                break
            next_size = max(current_size - decrement, num_features)
            self.hidden_layers.append(nn.Linear(current_size, next_size))
            self.batch_norm_layers.append(nn.BatchNorm1d(next_size))  # 添加 BatchNorm 层
            current_size = next_size


        self.hidden_layers = nn.ModuleList(self.hidden_layers)
        self.batch_norm_layers = nn.ModuleList(self.batch_norm_layers)
        self.output_layer = nn.Linear(current_size, num_features)  # 输出层

    def forward(self, x):
        for layer, batch_norm in zip(self.hidden_layers, self.batch_norm_layers):
            x = layer(x)
            x = batch_norm(x)
            x = torch.relu(x)
        x = self.output_layer(x)
        return x

class mpi_adapter(nn.Module):
    def __init__(self, embedding_dim, ffn_dim, num_layers =39):
        super(mpi_adapter, self).__init__()
        # self.embedding = nn.Embedding(num_embeddings=22, embedding_dim=embedding_dim)
        # self.par_position = PositionalEncoding(num_hiddens=embedding_dim, max_len = par_len)
        # self.mut_position = PositionalEncoding(num_hiddens=embedding_dim, max_len = mut_len)
        # self.mut_esm, _ = esm.pretrained.esm2_t33_650M_UR50D()
        # _, self.alphabet = esm.pretrained.esm2_t33_650M_UR50D()
        # self.padding_idx = self.alphabet.padding_idx
        # self.esm.to(self.device)
        # self.esm.eval()
        self.liner_mut = nn.Linear(1280,embedding_dim)
        self.liner_par = nn.Linear(1280,embedding_dim)
        # self.batch_converter = self.alphabet.get_batch_converter()
        self.layers = nn.ModuleList()
        for _ in range(num_layers):
            self.layers.append(InteractionBlock(embedding_dim, ffn_dim))
    def forward(self, mut0, mut1, par, mut0_padding_mask, par_padding_mask):
        # print([(str(0), sequence) for sequence in mut0s])
        # print([(str(0), sequence) for sequence in mut1s])
        # print([(str(0), sequence) for sequence in pars])
        # _, _, mut0 = self.batch_converter([(str(0), sequence) for sequence in mut0s])
        # _, _, mut1 = self.batch_converter([(str(0), sequence) for sequence in mut1s])
        # _, _, par = self.batch_converter([(str(0), sequence) for sequence in pars])
        # print(par)
        # mut0 = mut0s.to(self.device)
        # mut1 = mut1s.to(self.device)
        # par = pars.to(self.device)
        # mut0_padding_mask = mut0_padding_mask.to(self.device)
        # par_padding_mask = par_padding_mask.to(self.device)
        # mut0_padding_mask = mut0.eq(self.padding_idx)
        # mut1_padding_mask = mut1.eq(self.padding_idx)
        # par_padding_mask = par.eq(self.padding_idx)
        # mut0_padding_mask = torch.cat((mut0_padding_mask,mut1_padding_mask),dim=1)
        mut0 = torch.cat((mut0,mut1),dim=1)
        mut0 = self.liner_mut(mut0)
        par = self.liner_par(par)
        for layer in self.layers:
            mut0, par = layer(mut0, par, x_attn_padding_mask = mut0_padding_mask, r_attn_padding_mask = par_padding_mask)
        return par

class MLP_head(nn.Module):
    def __init__(self, embedding_dim, num_layers = 2):
        super(MLP_head, self).__init__()
        self.layers = nn.ModuleList()
        for _ in range(num_layers):
            self.layers.append(GroundingAttention(embedding_dim))
    def forward(self, x):
        for layer in self.layers:
            x = layer(x, x)
        return x[:, 0, :]

class MLP_head_one_layer(nn.Module):
    def __init__(self, embedding_dim, num_layers = 1,attention_heads = 16):
        super(MLP_head_one_layer, self).__init__()
        self.layers = nn.ModuleList()
        self.attention = torch.nn.MultiheadAttention(embed_dim = embedding_dim, num_heads = attention_heads, dropout = 0.0)
        # for _ in range(num_layers):
        #     self.layers.append(TransformerLayer(
        #         embedding_dim,
        #         4 * embedding_dim, # embed_dim = 1280
        #         attention_heads, # 20
        #         add_bias_kv=False,
        #         use_esm1b_layer_norm=True,
        #         use_rotary_embeddings=True,
        #         ))
    def forward(self, x, x_attn_padding_mask, need_head_weights=False):
        result = x[:, 0:1, :]
        x = x.transpose(0, 1)
        result = result.transpose(0, 1)
        result, attn = self.attention(result, x, x, key_padding_mask = x_attn_padding_mask)
        # for layer in self.layers:
        #     x = x.transpose(0, 1)
        #     x, _ = layer(x,
        #                 self_attn_padding_mask=x_attn_padding_mask,
        #                 need_head_weights=need_head_weights,
        #                 )
        #     x = x.transpose(0, 1)
        result = result.transpose(0, 1)
        result = result.squeeze(1)
        return result

class TAPPI(nn.Module):
    def __init__(self, embedding_dim = 192, num_layers = 39):
        super(TAPPI, self).__init__()
        self.result = nn.Parameter(torch.randn(1280))
        self.backbone = mpi_adapter(embedding_dim, embedding_dim*4, num_layers = num_layers)
        self.neck = MLP_head_one_layer(embedding_dim)
        # self.head = cross_entropy_bert(embedding_dim, device)
        self.head = nn.Linear(embedding_dim, 4)
    def forward(self,  mut0s, mut1s, pars, mut0_padding_mask = None, par_padding_mask = None):
        res = self.result.repeat(pars.shape[0], 1).unsqueeze(1) 
        pars = torch.concat([res,pars], dim = 1)
        false_column = torch.zeros(par_padding_mask.size(0), 1, dtype=torch.bool, device=pars.device)

        par_padding_mask = torch.cat((false_column, par_padding_mask), dim=1)
        x = self.backbone(mut0s, mut1s, pars, mut0_padding_mask, par_padding_mask)

        # x = self.neck(pars, par_padding_mask)
        x = self.neck(x, par_padding_mask)

        # x = self.head(label, x, weight)
        x = self.head(x)
        return x

In [ ]:
import torch

import torch.nn as nn
import math
from functools import partial
from esm.modules import ContactPredictionHead, ESM1bLayerNorm, RobertaLMHead, TransformerLayer, MultiheadAttention 
import esm
from typing import Union
import numpy as np

In [ ]:
class GHMC_Loss(nn.Module):
    def __init__(self, device, bins=8, momentum=0.3):
        super(GHMC_Loss, self).__init__()
        self.device = device
        self.bins = bins
        self.momentum = momentum
        self.edges = [float(x) / self.bins for x in range(self.bins + 1)]
        if momentum > 0:
            self.acc_sum = np.zeros(bins)

    def forward(self, targets, logits, class_weights=None):
        targets = torch.tensor(targets)
        targets = F.one_hot(targets, num_classes=4).float().to(self.device)
    
        edges = self.edges
        mmt = self.momentum
        weights = torch.zeros_like(logits)
        g = torch.abs(logits.softmax(dim=1).detach().to(self.device) - targets.to(self.device))
    
        tot = logits.shape[0] * logits.shape[1]
        n = 0  
        for i in range(self.bins):
            inds = (g >= edges[i]) & (g < edges[i + 1])
            num_in_bin = inds.sum().item()
            if num_in_bin > 0:
                if mmt > 0:
                    self.acc_sum[i] = mmt * self.acc_sum[i] + (1 - mmt) * num_in_bin
                    weights[inds] = tot / self.acc_sum[i]
                else:
                    weights[inds] = tot / num_in_bin
                n += 1
        if n > 0:
            weights = weights / n
    
        targets = targets.argmax(dim=1)
    
        if class_weights is not None:
            loss = F.cross_entropy(logits, targets, weight=class_weights, reduction='none')
        else:
            loss = F.cross_entropy(logits, targets, reduction='none')
    
        weights = weights.max(dim=1)[0]
        loss = (loss * weights).sum() / tot
    
        return loss

In [ ]:
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
device = 'cuda'

accum_steps = 10
batch_size = 8

import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
from tqdm import tqdm
import torch.nn.functional as F
import json


class TAPPI_Dataset(Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        sample = self.df.iloc[idx]

        return (
            sample['mut0_f'], 
            sample['mut1_f'], 
            sample['par0_f'], 
            sample['label']
        )


def tappi_collate_fn(batch):


    def pad_features(feature_list):
        max_len = max(f.shape[0] for f in feature_list)
        dim = feature_list[0].shape[1]
        padded, mask = [], []

        for f in feature_list:
            pad_len = max_len - f.shape[0]
            padded_f = np.pad(f, ((0, pad_len), (0, 0)), mode='constant', constant_values=0)
            padded.append(padded_f)

            m = np.concatenate([np.zeros(f.shape[0]), np.ones(pad_len)]).astype(bool)
            mask.append(m)

        padded_tensor = torch.tensor(np.stack(padded), dtype=torch.float32)
        mask_tensor = torch.tensor(np.stack(mask), dtype=torch.bool)
        return padded_tensor, mask_tensor
    mut0_list, mut1_list, par0_list, labels = zip(*batch)

    # padding & mask
    mut0, mut0_mask = pad_features(mut0_list)
    mut1, mut1_mask = pad_features(mut1_list)
    par0, par0_mask = pad_features(par0_list)

    labels = torch.tensor(labels, dtype=torch.long)

    return mut0, mut1, par0, mut0_mask, mut1_mask, par0_mask, labels


In [ ]:
from sklearn.model_selection import StratifiedKFold, train_test_split
import os, json, numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

n_splits = 5
num_epochs = 100
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

device = 'cuda'
def compute_accuracy(logits, labels):
    """
    logits: Tensor of shape [N, num_classes]
    labels: Tensor of shape [N]
    """
    preds = torch.argmax(logits, dim=1)
    correct = (preds == labels).sum().item()
    return correct / labels.size(0)
for fold, (train_val_idx, test_idx) in enumerate(
    skf.split(df, df["label"])
):
    print(f"\n========== Fold {fold} ==========")

    # ---------- folder ----------
    fold_dir = f"./cv_results/fold_{fold}"
    os.makedirs(fold_dir, exist_ok=True)

    # ---------- split ----------
    train_val_df = df.iloc[train_val_idx].reset_index(drop=True)
    test_fold_df = df.iloc[test_idx].reset_index(drop=True)

    labels = train_val_df['label']

    
    val_df, test_df = train_test_split(
        test_fold_df,
        test_size=0.5,
        random_state=42,
        stratify=test_fold_df["label"]
    )
    
    # ---------- dataset ----------
    train_dataset = TAPPI_Dataset(train_val_df)
    val_dataset   = TAPPI_Dataset(val_df)
    test_dataset  = TAPPI_Dataset(test_df)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=tappi_collate_fn,
        num_workers=8
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=tappi_collate_fn,
        num_workers=8
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=tappi_collate_fn,
        num_workers=8
    )
    model = TAPPI(num_layers=39).to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=4e-5,
        betas=(0.9, 0.999),
        weight_decay=1e-2
    )

    model_loss = GHMC_Loss(device)

    # class_weights_torch = torch.tensor(
    #     [class_weights[i] for i in range(4)],
    #     dtype=torch.float32
    # ).to(device)

    best_val_acc = 0.0
    best_ckpt_path = os.path.join(fold_dir, "best_model.pth")

    train_acc_list, val_acc_list = [], []
    for ep in range(20):
        # -------- TRAIN --------
        model.train()
        all_preds, all_labels = [], []

        for step, (mut0, mut1, par0, mut0_mask, mut1_mask, par0_mask, labels) in tqdm(
            enumerate(train_loader),
            total=len(train_loader),
            desc=f"Fold {fold} | Epoch {ep+1}/{num_epochs}"
        ):
            mut0, mut1, par0 = mut0.to(device), mut1.to(device), par0.to(device)
            mut0_mask, mut1_mask, par0_mask = (
                mut0_mask.to(device),
                mut1_mask.to(device),
                par0_mask.to(device),
            )
            labels = labels.to(device)

            preds = model(
                mut0, mut1, par0,
                torch.cat([mut0_mask, mut1_mask], dim=1),
                par0_mask
            )

            loss = model_loss(labels, preds, None)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            all_preds.append(preds.detach().cpu())
            all_labels.append(labels.detach().cpu())

        train_acc = compute_accuracy(
            torch.cat(all_preds),
            torch.cat(all_labels)
        )
        train_acc_list.append(train_acc)

        # -------- VALIDATION --------
        model.eval()
        all_preds, all_labels = [], []

        with torch.no_grad():
            for mut0, mut1, par0, mut0_mask, mut1_mask, par0_mask, labels in val_loader:
                mut0, mut1, par0 = mut0.to(device), mut1.to(device), par0.to(device)
                mut0_mask, mut1_mask, par0_mask = (
                    mut0_mask.to(device),
                    mut1_mask.to(device),
                    par0_mask.to(device),
                )
                labels = labels.to(device)

                preds = model(
                    mut0, mut1, par0,
                    torch.cat([mut0_mask, mut1_mask], dim=1),
                    par0_mask
                )

                all_preds.append(preds.cpu())
                all_labels.append(labels.cpu())

        val_acc = compute_accuracy(
            torch.cat(all_preds),
            torch.cat(all_labels)
        )
        val_acc_list.append(val_acc)

        # -------- save best --------
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_ckpt_path)
        with open(os.path.join(fold_dir, "val_acc_list.json"), 'w') as f:
            json.dump(val_acc_list, f)
        with open(os.path.join(fold_dir, "train_acc_list.json"), 'w') as f:
            json.dump(train_acc_list, f)
            
        print(f"Epoch {ep+1}: Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}")

